# MacroAnchor G10 10Y Sovereign RV — V3.2 credit-stratified research extension

**STATUS: unexecuted on historical sovereign CDS/rating files.** This is a *new, post-V2.3* predeclared research test, not additional historical evidence for V2.3. The original V2.3 is kept intact in a separate notebook in this repository.

**Run all in Google Colab.** The **first code cell immediately opens a single upload dialog**. Select together:

- **Required:** `g10_panel_model_objects.csv` (frozen model panel) **and either** multiple **raw Investing.com 5Y sovereign CDS CSV exports** (e.g. `DEGV5YUSAC Overview.csv`, `JPGV5YUSAC Overview.csv`) **or** one premerged `sovereign_cds_5y.csv` / `cds.parquet` historical panel. Raw exports are parsed and merged automatically before preflight; filenames must retain the ticker or country. No current snapshot.
- **Optional:** `sovereign_ratings_history.csv` (actual dated, global-scale sovereign local-currency rating records), `expanded_macroanchor_model_objects.csv` (additional countries with the **same frozen signal-generation definitions**), `corporate_credit_factors.csv` (Moody's Aaa/Baa or ICE BofA corporate OAS time series; used **only as aggregate credit-cycle factors**, never as country CDS), or the six FRED bundles when an original model panel is unavailable.

All files must be selected **at the first dialog**. Once the upload finishes: **no further file prompts or network requests**. Missing required CDS history causes an immediate descriptive error, not a silent fallback to a synthetic proxy.

**Sources/instructions**: [François Geerolf, sovereign 5Y CDS panel](https://fgeerolf.com/data/investing/cds.html) advertises historical daily `cds.parquet` from approximately 2008 (check download access/licence yourself); [FRED Moody's Aaa](https://fred.stlouisfed.org/series/AAA) and [Baa](https://fred.stlouisfed.org/series/BAA); [FRED ICE BofA corporate OAS](https://fred.stlouisfed.org/series/BAMLC0A4CBBB). ICE BofA OAS series have a trailing ~3-year public FRED window from April 2026; corporate OAS are **not sovereign spreads**. Downloading web data is deliberately left outside the notebook.

**V3 decisions frozen before seeing V3 results**: (A) absolute 5Y CDS bands <40 bp / 40–100 bp / ≥100 bp, L/S extreme MacroAnchor score within each eligible band (1 each); (B) aggregate equal *gross notional* budgets across active bands; (C) historical rating-bucket analogous test *only if dated ratings are supplied*; (D) matched-CDS pair with ≤25 bp within-pair CDS difference as a sensitivity; (E) common-sample V2.3 comparisons, 12m forward return NW(12), nonoverlap track diagnostics. No threshold grid search or post-hoc strategy promotion. A band with <2 sovereigns at a date is inactive.

**Not claimed**: DV01 neutrality does not establish CS01 neutrality; matching 5Y CDS does not equal a 10Y CDS hedge. The returns are local-currency *duration-based proxies*, not a priced FX-forward-hedged, marked-to-market tradable strategy.

In [ ]:
# ===== INPUT STEP — runs first and blocks immediately for upload =====
# Required together: g10_panel_model_objects.csv and raw Investing.com CDS CSVs (or one merged historical CDS file).
# Optional: sovereign ratings history, corporate OAS controls, expanded MacroAnchor panel.
# After this cell completes, the notebook will not ask for any further input.

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError("This notebook's upload-first workflow is intended for Google Colab.") from exc

print("UPLOAD NOW: select BOTH the frozen MacroAnchor panel and historical sovereign 5Y CDS.")
print("Required: g10_panel_model_objects.csv + multiple original Investing.com CDS CSVs (or one cds.parquet / sovereign_cds_5y.csv)")
print("Optional in this same upload: dated ratings, corporate OAS controls, expanded MacroAnchor panel.")
print("If no frozen panel: raw snapshot or six FRED bundles can replace the panel file.")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No files uploaded. Upload the required input files before continuing.")

UPLOADED_NAMES = list(uploaded.keys())
print("\nUpload complete. From this point onward the notebook is unattended.")
print("Uploaded files:")
for name in UPLOADED_NAMES:
    print(" -", name)


In [ ]:
# ===== INVESTING.COM RAW CDS CSV BRIDGE — insert AFTER FIRST UPLOAD, BEFORE PREFLIGHT =====
# Accept several original Investing.com exports (one CDS 5Y sovereign per country).
# Converts them, without additional prompts, into sovereign_cds_5y.csv expected by V3.
# Parent MacroAnchor strategy, signals, portfolio rules, and statistics are unchanged.

from pathlib import Path
import re
import unicodedata
import hashlib
import json
import pandas as pd

_INV_ISO2 = {
    'DE':'DEU','US':'USA','GB':'GBR','JP':'JPN','CH':'CHE',
    'AU':'AUS','CA':'CAN','SE':'SWE','NO':'NOR','NZ':'NZL',
    'FR':'FRA','IT':'ITA','ES':'ESP','PT':'PRT','BE':'BEL',
    'NL':'NLD','AT':'AUT','FI':'FIN','IE':'IRL','DK':'DNK',
    'KR':'KOR','MX':'MEX','BR':'BRA',
}
_INV_NAMES = {
    'germany':'DEU','allemagne':'DEU','deutschland':'DEU',
    'united states':'USA','usa':'USA','etats unis':'USA',
    'united kingdom':'GBR','great britain':'GBR','uk':'GBR',
    'japan':'JPN','japon':'JPN','switzerland':'CHE','suisse':'CHE',
    'australia':'AUS','australie':'AUS','canada':'CAN',
    'sweden':'SWE','suede':'SWE','norway':'NOR','norvege':'NOR',
    'new zealand':'NZL','france':'FRA','italy':'ITA','italia':'ITA',
    'spain':'ESP','espana':'ESP','espagne':'ESP','portugal':'PRT',
    'belgium':'BEL','netherlands':'NLD','austria':'AUT',
    'finland':'FIN','ireland':'IRL','denmark':'DNK','south korea':'KOR',
    'mexico':'MEX','brazil':'BRA',
}

def _inv_country_from_filename(filename):
    """Require a country in the original filename: ticker or explicit country name."""
    name = Path(filename).stem.upper()
    codes = re.findall(r'(?<![A-Z0-9])([A-Z]{2})GV5Y[A-Z0-9=]*(?![A-Z0-9])', name)
    mapped = {_INV_ISO2[c] for c in codes if c in _INV_ISO2}
    if len(mapped) > 1:
        raise ValueError(f'Ambiguous country ticker in filename: {filename}')
    if mapped:
        return next(iter(mapped))
    norm = ''.join(ch for ch in unicodedata.normalize('NFKD',name.lower())
                   if not unicodedata.combining(ch))
    norm = re.sub(r'[^a-z0-9]+', ' ', norm).strip()
    matches = {iso for label, iso in _INV_NAMES.items()
               if re.search(r'(?<![a-z0-9])' + re.escape(label) + r'(?![a-z0-9])', norm)}
    if len(matches) == 1:
        return next(iter(matches))
    # Explicit ISO3 naming also supported: DEU_CDS5Y_history.csv
    if ('cds' in norm or '5y' in norm) and norm.split(' ')[0].upper() in set(_INV_ISO2.values()):
        return norm.split(' ')[0].upper()
    raise ValueError(f'Cannot identify one sovereign from filename {filename!r}. '
                     'Preserve the Investing.com ticker (e.g. DEGV5YUSAC Overview.csv), '
                     'or rename it as DEU_CDS5Y_raw.csv.')

def _inv_raw_signature(path):
    """Check header only; never mistake the frozen model, ratings or OAS for CDS quotes."""
    if path.suffix.lower() != '.csv':
        return False
    try:
        header = pd.read_csv(path, nrows=0, encoding='utf-8-sig').columns
        names = {str(c).strip().lower() for c in header}
        return {'date','price'}.issubset(names) and bool(
            names.intersection({'open','high','low','change %','change%'}))
    except Exception:
        return False

def _inv_price_bp(series):
    raw = series.astype(str).str.strip().str.replace('\u00a0', '', regex=False)
    raw = raw.str.replace(r'(?i)\s*(bp|bps|basis\s*points)\s*$', '', regex=True)
    # Investing.com export: 12.51, 1,234.56, or 1.234,56 depending on locale.
    def parse(v):
        v = re.sub(r'\s+', '', v)
        if v in {'', '-', '--', 'nan', 'None'}: return float('nan')
        if ',' in v and '.' in v:
            v = v.replace(',', '') if v.rfind('.') > v.rfind(',') else v.replace('.', '').replace(',', '.')
        elif ',' in v:
            if re.fullmatch(r'[+-]?\d{1,3}(,\d{3})+', v): v = v.replace(',', '')
            else: v = v.replace(',', '.')
        try: return float(v)
        except ValueError: return float('nan')
    return pd.to_numeric(raw.map(parse), errors='coerce')

def _inv_date(series, filename):
    raw = series.astype(str).str.strip()
    # US-style dates are what the actual DEGV5YUSAC export uses.
    candidates = [(f, pd.to_datetime(raw, format=f, errors='coerce'))
                  for f in ('%m/%d/%Y', '%Y-%m-%d', '%d/%m/%Y')]
    best_score = max(s.notna().sum() for _, s in candidates)
    best = [(fmt, s) for fmt, s in candidates if s.notna().sum() == best_score]
    if best_score < max(1, int(.95 * len(raw))):
        raise ValueError(f'{filename}: failed to parse >=95% of Investing.com dates')
    # If two date conventions fit equally well and give different dates, refuse silent ambiguity.
    if len(best) > 1 and any(not best[0][1].equals(s) for _, s in best[1:]):
        raise ValueError(f'{filename}: ambiguous date format. Expected MM/DD/YYYY, like German example.')
    return best[0][1]

def _inv_parse_file(path):
    country = _inv_country_from_filename(path.name)
    raw = pd.read_csv(path, encoding='utf-8-sig')
    columns = {str(c).strip().lower(): c for c in raw.columns}
    dates = _inv_date(raw[columns['date']], path.name)
    spread = _inv_price_bp(raw[columns['price']])
    invalid = dates.isna() | spread.isna() | (spread < 0) | (spread >= 3000)
    if invalid.mean() > .01:
        raise ValueError(f'{path.name}: {invalid.sum()} invalid rows ({invalid.mean():.1%}), '
                         'check locale/units before continuing.')
    df = pd.DataFrame({'date': dates.loc[~invalid], 'country': country,
                       'cds5y_bp': spread.loc[~invalid]})
    if df['date'].dt.to_period('M').nunique() < 12:
        raise ValueError(f'{path.name}: fewer than 12 historical months; snapshot/short export rejected.')
    df = df.sort_values('date').drop_duplicates(['country', 'date'], keep='last')
    print(f'  {path.name} -> {country}: {len(df):,} daily quotes; '
          f'{df.date.min():%Y-%m-%d} to {df.date.max():%Y-%m-%d}')
    return df

# Work ONLY with explicitly uploaded files. No downloading, searching or later dialogs.
_inv_raw_files = [Path(name) for name in UPLOADED_NAMES if _inv_raw_signature(Path(name))]
if _inv_raw_files:
    _inv_other_cds = [name for name in UPLOADED_NAMES
                      if name not in {p.name for p in _inv_raw_files}
                      and ('cds' in name.lower() or 'sovereign_spread' in name.lower())
                      and Path(name).suffix.lower() in {'.csv','.parquet','.pq'}]
    if _inv_other_cds:
        raise ValueError('Upload EITHER raw Investing.com files OR one pre-merged CDS history, not both: '
                         + ', '.join(_inv_other_cds))
    print(f'Converting {len(_inv_raw_files)} Investing.com raw CDS exports:')
    _inv_all = pd.concat([_inv_parse_file(p) for p in _inv_raw_files], ignore_index=True)
    dup = _inv_all.duplicated(['country','date'], keep=False)
    if dup.any():
        diffs = _inv_all.loc[dup].groupby(['country','date'])['cds5y_bp'].agg(lambda x: x.max()-x.min())
        if (diffs > 1e-6).any():
            raise ValueError('Conflicting CDS quotes for the same country/date across uploaded files. '
                             'Remove duplicates or compare instrument currencies.')
    _inv_all = _inv_all.sort_values(['country','date']).drop_duplicates(['country','date'])
    _inv_out = Path('sovereign_cds_5y.csv')
    _inv_all.to_csv(_inv_out, index=False, date_format='%Y-%m-%d', float_format='%.6f')
    _inv_manifest = {
        'source': 'Investing.com user-exported sovereign CDS 5Y daily CSV',
        'files': [{'filename': p.name,
                   'sha256': hashlib.sha256(p.read_bytes()).hexdigest(),
                   'country': _inv_country_from_filename(p.name)} for p in _inv_raw_files],
        'normalized_file': _inv_out.name,
        'normalized_sha256': hashlib.sha256(_inv_out.read_bytes()).hexdigest(),
        'normalized_rows': len(_inv_all),
        'normalized_countries': sorted(_inv_all.country.unique().tolist()),
    }
    Path('investing_raw_cds_sources.json').write_text(json.dumps(_inv_manifest,indent=2))
    # Hide raw files from V3's one-CDS-file preflight, expose only the canonical merged file.
    UPLOADED_NAMES = [n for n in UPLOADED_NAMES if n not in {p.name for p in _inv_raw_files}]
    UPLOADED_NAMES.append(_inv_out.name)
    print('Generated:', _inv_out, '|', len(_inv_all), 'quotes |',
          _inv_all.country.nunique(), 'countries; no further input required.')
else:
    print('No Investing.com raw exports detected; using existing normalized CDS input unchanged.')


In [ ]:
# ===== UPLOAD NAME HOTFIX — after Investing.com parser, before IMMEDIATE INPUT PREFLIGHT =====
# Colab saves re-uploaded files as "g10_panel_model_objects (4).csv" (etc.).
# The legacy V2/V3 loader expects the exact canonical name; create an exact alias
# ONLY from the file explicitly selected in the opening upload dialog.
# No extra prompts, no web requests, no strategy/model changes.
from pathlib import Path
import re
import shutil


def _colab_canonical_basename(filename):
    return re.sub(r'\s*\(\d+\)(?=\.[^.]+$)', '', Path(filename).name)


for _target in ('g10_panel_model_objects.csv', 'g10_panel_monthly_snapshot.csv'):
    _candidates = [name for name in UPLOADED_NAMES
                   if _colab_canonical_basename(name).lower() == _target.lower()]
    if len(_candidates) > 1:
        raise RuntimeError(
            f"Multiple uploaded files correspond to {_target}: {_candidates}. "
            "Select only one version in the initial upload."
        )
    if _candidates:
        _original = Path(_candidates[0])
        if not _original.is_file():
            raise FileNotFoundError(f"Uploaded panel cannot be read: {_original}")
        _canonical = Path(_target)
        if _original.name != _canonical.name:
            # Overwrite potential stale canonical file from a previous Colab run.
            shutil.copyfile(_original, _canonical)
            # Replace the uploaded spelling in the in-memory input list so the
            # code is idempotent (running this cell twice is harmless).
            UPLOADED_NAMES = [_target if n == _original.name else n for n in UPLOADED_NAMES]
            print(f"Colab input alias: {_original.name!r} -> {_target!r}")
        else:
            print(f"Canonical panel input: {_target}")

# The original preflight below may now check exact filenames; the loader will
# read the canonical alias. The SHA check in load_model_panel remains active.


In [ ]:
# ===== IMMEDIATE INPUT PREFLIGHT — before long computations =====
from pathlib import Path
import pandas as pd
import json, hashlib

UPLOAD_PATHS = [Path(n) for n in UPLOADED_NAMES]
def is_uploaded_cds(p):
    return (p.suffix.lower() in {'.csv','.parquet','.pq'} and
            ('cds' in p.name.lower() or 'sovereign_spread' in p.name.lower()))
CDS_FILES = [p for p in UPLOAD_PATHS if is_uploaded_cds(p)]
if len(CDS_FILES) != 1:
    raise RuntimeError(
        'V3 requires one combined sovereign CDS5Y file after automatic raw-file merging. '
        'Expected Investing.com raw CSV files or a sovereign_cds_5y.csv / cds.parquet historical panel (not a snapshot). '
        'Files detected: '+', '.join(map(str,CDS_FILES))
    )
if not any(p.name=='g10_panel_model_objects.csv' for p in UPLOAD_PATHS) and not any(
    p.name=='g10_panel_monthly_snapshot.csv' for p in UPLOAD_PATHS
) and not any(p.name.lower().startswith('fredgraph') for p in UPLOAD_PATHS):
    raise RuntimeError('First upload must also include frozen g10 model panel, raw snapshot, or six FRED bundles.')
CDS_PATH=CDS_FILES[0]
print('PREFLIGHT PASS: historical sovereign CDS candidate =', CDS_PATH)
print('No additional upload will be requested.')


In [ ]:
# ===== 0. Configuration =====
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import io, json, hashlib, warnings, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.5f}")
plt.rcParams.update({"figure.dpi": 115, "font.size": 9, "axes.grid": True, "grid.alpha": 0.25})

START = pd.Timestamp("1960-01-01")
PANEL_END = pd.Timestamp("2026-05-31")
MINW = 120
H = 12
NW_LAG = 12
PI_HL = 36
RSTAR_HL = 120
ANCHOR_INITIALIZATION = "post_ewma_bfill"
QUARTERLY_CPI_RELEASE_LAG_MONTHS = 1

# Frozen strategy choices. Do not retune after seeing V2.3 results.
HEADLINE_K = 3
SENSITIVITY_K = 2
MIN_COUNTRIES = 6
COST_GRID_BP = [0.0, 1.0, 2.0, 5.0, 10.0, 25.0]
FX_HEDGE_ANNUAL_DRAG_BP = [0.0, 5.0, 10.0, 20.0]  # transparent stress only, not an estimate

MODEL_PANEL_FILENAME = "g10_panel_model_objects.csv"
MODEL_PANEL_EXPECTED_SHA = "1628c4eaff487f6d87cc93fe62c7af909850d9ceedf448d9d8321f8b8477bbf6"
SNAPSHOT_FILENAME = "g10_panel_monthly_snapshot.csv"
OUT_DIR = Path("macroanchor_g10_rv_v2_outputs")
OUT_DIR.mkdir(exist_ok=True)

# Data behavior: inputs come ONLY from the first upload window; no runtime FRED calls.

PANEL_SPEC = {
    "USA": {"y10": "IRLTLT01USM156N", "r3m": "IR3TIB01USM156N", "cpi": ["USACPIALLMINMEI", "CPIAUCSL"]},
    "DEU": {"y10": "IRLTLT01DEM156N", "r3m": "IR3TIB01DEM156N", "cpi": ["DEUCPIALLMINMEI"]},
    "GBR": {"y10": "IRLTLT01GBM156N", "r3m": "IR3TIB01GBM156N", "cpi": ["GBRCPIALLMINMEI"]},
    "JPN": {"y10": "IRLTLT01JPM156N", "r3m": "IR3TIB01JPM156N", "cpi": ["JPNCPIALLMINMEI"]},
    "CAN": {"y10": "IRLTLT01CAM156N", "r3m": "IR3TIB01CAM156N", "cpi": ["CANCPIALLMINMEI"]},
    "CHE": {"y10": "IRLTLT01CHM156N", "r3m": "IR3TIB01CHM156N", "cpi": ["CHECPIALLMINMEI"]},
    "SWE": {"y10": "IRLTLT01SEM156N", "r3m": "IR3TIB01SEM156N", "cpi": ["SWECPIALLMINMEI"]},
    "NOR": {"y10": "IRLTLT01NOM156N", "r3m": "IR3TIB01NOM156N", "cpi": ["NORCPIALLMINMEI"]},
    "AUS": {"y10": "IRLTLT01AUM156N", "r3m": "IR3TIB01AUM156N", "cpi": ["AUSCPIALLQINMEI"]},
    "NZL": {"y10": "IRLTLT01NZM156N", "r3m": "IR3TIB01NZM156N", "cpi": ["NZLCPIALLQINMEI"]},
}

print("Frozen V2.3 specification")
print("headline top/bottom:", HEADLINE_K, "| sensitivity:", SENSITIVITY_K)
print("panel cutoff:", PANEL_END.date(), "| forecast horizon:", H, "months")


In [ ]:
# ===== 1. Core model + inference helpers (same MacroAnchor panel logic) =====
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def locate_file(name):
    for p in [Path(name), Path('/content')/name, Path('/mnt/data')/name]:
        if p.exists(): return p
    for root in [Path('/content'), Path.cwd(), Path('/mnt/data')]:
        if root.exists():
            hits = list(root.rglob(name))
            if hits: return hits[0]
    return None

def vas_B(tau, kappa):
    kappa = max(float(kappa), 1e-4)
    return (1.0 - np.exp(-kappa * tau)) / kappa

def vas_yield(tau, short_rate, kappa, theta, sigma):
    kappa = max(float(kappa), 1e-4)
    B = vas_B(tau, kappa)
    lnA = (theta - sigma*sigma/(2*kappa*kappa))*(B-tau) - sigma*sigma*B*B/(4*kappa)
    return (-lnA + B*short_rate) / tau

def build_anchor(df, initialization=ANCHOR_INITIALIZATION):
    idx = pd.DatetimeIndex(df.index)
    short_s = pd.Series(df['r3m'].to_numpy(float), index=idx)
    cpi_s = pd.Series(df['cpi'].to_numpy(float), index=idx)
    pi_raw = cpi_s.pct_change(12, fill_method=None).ewm(halflife=PI_HL).mean()
    if initialization == 'draft_pre_rstar_bfill':
        pi = pi_raw.bfill(); rstar = (short_s-pi).ewm(halflife=RSTAR_HL).mean().bfill()
    elif initialization in ('post_ewma_bfill','strict'):
        pi = pi_raw; rstar = (short_s-pi).ewm(halflife=RSTAR_HL).mean()
        if initialization == 'post_ewma_bfill':
            pi = pi.bfill(); rstar = rstar.bfill()
    else: raise ValueError(initialization)
    return rstar + pi, rstar, pi

def past_only_z(x, minobs=MINW):
    x = np.asarray(x, float); z = np.full(len(x), np.nan)
    for t in range(minobs, len(x)):
        hist = x[:t]; hist = hist[np.isfinite(hist)]
        if len(hist) < minobs or not np.isfinite(x[t]): continue
        sd = hist.std(ddof=1)
        if sd > 1e-12: z[t] = (x[t] - hist.mean()) / sd
    return z

def duration_proxy_10y(y):
    """Paper-consistent 10Y annuity/duration proxy used by the international target."""
    y = np.maximum(np.asarray(y, float), 1e-4)
    return (1.0 - (1.0 + y)**-10) / y

def run_country_objects(df):
    df = df.sort_index().copy(); n=len(df)
    r=df['r3m'].to_numpy(float); y10=df['y10'].to_numpy(float)
    anchor,rstar,pi=build_anchor(df); anc=anchor.to_numpy(float)
    kappa=np.full(n,np.nan); sigma=np.full(n,np.nan); tp=np.full(n,np.nan)
    for t in range(MINW,n):
        g=anc[:t]-r[:t]; dr=r[1:t+1]-r[:t]
        m=np.isfinite(g)&np.isfinite(dr)
        if m.sum()<MINW: continue
        X=np.column_stack([np.ones(m.sum()),g[m]])
        a0,slope=np.linalg.lstsq(X,dr[m],rcond=None)[0]
        kappa[t]=float(np.clip(slope*12.0,1e-4,20.0))
        resid=dr[m]-(a0+slope*g[m]); sigma[t]=float(np.std(resid,ddof=0)*np.sqrt(12.0))
        tp[t]=y10[t]-vas_yield(10.0,r[t],kappa[t],anc[t],sigma[t])
    D=duration_proxy_10y(y10)
    rx=np.full(n,np.nan)
    for t in range(n-H):
        rx[t]=y10[t]-D[t]*(y10[t+H]-y10[t])-r[t]
    return pd.DataFrame({
        'date':df.index,'country':df['country'].iloc[0],'y10':y10,'r3m':r,
        'slope':y10-r,'pi_trend':pi.to_numpy(float),'rstar':rstar.to_numpy(float),
        'anchor':anc,'kappa':kappa,'sigma':sigma,'tp':tp,'tp_z_past':past_only_z(tp),
        'duration10':D,'rx':rx
    })

def nw_fit(y, X=None, lag=NW_LAG):
    y=np.asarray(y,float)
    if X is None: X=np.empty((len(y),0))
    X=np.asarray(X,float)
    if X.ndim==1: X=X[:,None]
    m=np.isfinite(y)&np.isfinite(X).all(axis=1)
    y=y[m]; X=X[m]
    X=np.column_stack([np.ones(len(y)),X])
    b=np.linalg.lstsq(X,y,rcond=None)[0]; e=y-X@b
    bread=np.linalg.pinv(X.T@X); score=X*e[:,None]; S=score.T@score
    for ell in range(1,min(lag,len(y)-1)+1):
        w=1-ell/(lag+1); G=score[ell:].T@score[:-ell]; S+=w*(G+G.T)
    V=bread@S@bread; se=np.sqrt(np.maximum(np.diag(V),0))
    den=np.sum((y-y.mean())**2); r2=1-np.sum(e*e)/den if den>0 else np.nan
    return b,se,V,r2,int(len(y))

In [ ]:
# ===== 2. Upload-only data layer: model panel -> snapshot -> six browser-downloaded FRED bundles =====
# IMPORTANT: no filesystem search and no web request occurs here.
# Only files explicitly selected in the first upload window are eligible inputs.

def monthly_cpi_from_source(cpi, series_id):
    cpi = cpi.dropna().sort_index().astype(float)
    if 'QINMEI' in str(series_id).upper():
        qend = cpi.index.to_period('Q').to_timestamp(how='end').normalize()
        release = qend + pd.offsets.MonthEnd(QUARTERLY_CPI_RELEASE_LAG_MONTHS)
        s = pd.Series(cpi.to_numpy(), index=release).groupby(level=0).last().sort_index()
        return s.resample('ME').last().ffill()
    return cpi.resample('ME').last()

def required_fred_series():
    ids = []
    for spec in PANEL_SPEC.values():
        ids.extend([spec['y10'], spec['r3m']])
        ids.extend(spec['cpi'])
    return list(dict.fromkeys(ids))

def _uploaded_path(name):
    p = Path(name)
    if not p.exists():
        raise FileNotFoundError(f"Uploaded file is not present in the Colab working directory: {name}")
    return p

def _read_candidate_fred_csv(name):
    """Read one uploaded CSV; return only recognized canonical FRED columns."""
    p = _uploaded_path(name)
    try:
        raw = pd.read_csv(p)
    except Exception:
        return None

    if raw.empty or raw.shape[1] < 2:
        return None

    required = set(required_fred_series())
    matching = [c for c in raw.columns if c in required]
    if not matching:
        return None

    date_col = next(
        (c for c in raw.columns if str(c).strip().lower() in {'observation_date','date'}),
        raw.columns[0],
    )
    dates = pd.to_datetime(raw[date_col], errors='coerce')
    keep = dates.notna()

    out = raw.loc[keep, matching].copy()
    out.index = pd.DatetimeIndex(dates.loc[keep])
    out.index.name = 'date'
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    return out.sort_index()

def merge_uploaded_fred_bundles():
    candidates = []
    for name in UPLOADED_NAMES:
        if not str(name).lower().endswith('.csv'):
            continue
        if Path(name).name in {MODEL_PANEL_FILENAME, SNAPSHOT_FILENAME}:
            continue
        frame = _read_candidate_fred_csv(name)
        if frame is not None:
            candidates.append((name, frame))

    if not candidates:
        raise RuntimeError(
            "None of the uploaded files contains the canonical FRED series. "
            "Upload the six fredgraph CSV files linked at the top of the notebook."
        )

    print("Recognized uploaded FRED bundles:")
    for name, frame in candidates:
        print(f"  {name}: {len(frame.columns)} canonical series")

    merged = pd.DataFrame()
    for name, frame in candidates:
        if merged.empty:
            merged = frame.copy()
            continue

        idx = merged.index.union(frame.index)
        merged = merged.reindex(idx)
        frame = frame.reindex(idx)

        for col in frame.columns:
            if col not in merged.columns:
                merged[col] = frame[col]
            else:
                both = merged[col].notna() & frame[col].notna()
                if both.any():
                    diff = (merged.loc[both, col] - frame.loc[both, col]).abs()
                    if (diff > 1e-12).any():
                        bad = diff[diff > 1e-12].index[0]
                        raise ValueError(f"Conflicting uploaded values for {col} at {bad.date()}.")
                merged[col] = merged[col].combine_first(frame[col])

    # Validate country-by-country rather than demanding both alternative US CPI series.
    missing = {}
    for country, spec in PANEL_SPEC.items():
        miss = []
        for sid in [spec['y10'], spec['r3m']]:
            if sid not in merged.columns or merged[sid].notna().sum() < 24:
                miss.append(sid)
        if not any(sid in merged.columns and merged[sid].notna().sum() >= 24 for sid in spec['cpi']):
            miss.append("one_of:" + "|".join(spec['cpi']))
        if miss:
            missing[country] = miss

    if missing:
        raise RuntimeError(
            "Uploaded FRED bundles are incomplete. Missing canonical inputs:\n"
            + json.dumps(missing, indent=2)
        )

    print("Uploaded FRED coverage validation: PASS")
    return merged.sort_index()

def build_raw_snapshot_from_uploaded_bundles():
    wide = merge_uploaded_fred_bundles()
    rows = []
    failures = {}

    for country, spec in PANEL_SPEC.items():
        try:
            cpi_id = next(
                sid for sid in spec['cpi']
                if sid in wide.columns and wide[sid].notna().sum() >= 24
            )

            y = wide[spec['y10']].dropna().resample('ME').last() / 100.0
            r = wide[spec['r3m']].dropna().resample('ME').last() / 100.0
            cp = monthly_cpi_from_source(wide[cpi_id], cpi_id)

            d = pd.concat(
                [y.rename('y10'), r.rename('r3m'), cp.rename('cpi')],
                axis=1
            )
            d = d.loc[(d.index >= START) & (d.index <= PANEL_END)].dropna()

            if len(d) < MINW + H + 24:
                raise RuntimeError(f"only {len(d)} complete months")

            z = d.reset_index()
            z = z.rename(columns={z.columns[0]: 'date'})
            z.insert(1, 'country', country)
            rows.append(z)
            print(country, "OK", len(d), d.index.min().date(), "->", d.index.max().date())

        except Exception as exc:
            failures[country] = repr(exc)
            print(country, "FAILED", exc)

    if len(rows) < MIN_COUNTRIES:
        raise RuntimeError(
            "Uploaded data did not assemble enough countries.\n"
            + json.dumps(failures, indent=2)
        )

    raw = pd.concat(rows, ignore_index=True).sort_values(['country','date'])
    raw.to_csv(SNAPSHOT_FILENAME, index=False)
    print(f"Frozen raw snapshot written to {SNAPSHOT_FILENAME}")
    return raw

def load_model_panel():
    uploaded_basenames = {Path(x).name: x for x in UPLOADED_NAMES}

    # 1) Exact frozen model panel uploaded in the initial window.
    if MODEL_PANEL_FILENAME in uploaded_basenames:
        path = _uploaded_path(uploaded_basenames[MODEL_PANEL_FILENAME])
        got = sha256_file(path)
        if got == MODEL_PANEL_EXPECTED_SHA:
            print("Using uploaded frozen V7 model panel; SHA verified.")
        else:
            warnings.warn(
                "Uploaded model-panel SHA differs from archived V7 hash; "
                "continuing but recording provenance."
            )
        p = pd.read_csv(path, parse_dates=['date']).sort_values(['country','date']).reset_index(drop=True)
        if 'duration10' not in p:
            p['duration10'] = duration_proxy_10y(p['y10'])
        return p, {'source':'uploaded_model_panel','path':str(path),'sha256':got}

    # 2) Exact frozen raw snapshot uploaded in the initial window.
    if SNAPSHOT_FILENAME in uploaded_basenames:
        path = _uploaded_path(uploaded_basenames[SNAPSHOT_FILENAME])
        print("Using uploaded frozen raw snapshot:", path)
        raw = pd.read_csv(path, parse_dates=['date']).sort_values(['country','date'])
        source = {'source':'uploaded_raw_snapshot','path':str(path),'sha256':sha256_file(path)}

    # 3) Otherwise the six browser-downloaded FRED bundles from the same upload window.
    else:
        print("No frozen panel/snapshot uploaded; building from the uploaded six FRED bundles.")
        raw = build_raw_snapshot_from_uploaded_bundles()
        source = {
            'source':'uploaded_fred_six_bundle',
            'path':SNAPSHOT_FILENAME,
            'sha256':sha256_file(SNAPSHOT_FILENAME),
        }

    objs = []
    for country, g in raw.groupby('country'):
        d = g.sort_values('date').set_index('date').copy()
        d['country'] = country
        objs.append(run_country_objects(d))

    p = pd.concat(objs, ignore_index=True).sort_values(['country','date']).reset_index(drop=True)
    p.to_csv(MODEL_PANEL_FILENAME, index=False)
    source['generated_model_panel_sha256'] = sha256_file(MODEL_PANEL_FILENAME)
    return p, source

model_panel, DATA_PROVENANCE = load_model_panel()
print('rows:', len(model_panel), '| raw countries:', model_panel.country.nunique())
analysis = model_panel.dropna(subset=['rx','tp_z_past','y10']).copy()
print('usable:', len(analysis), 'country-months |', analysis.country.nunique(), 'countries |', analysis.date.nunique(), 'months')
print('provenance:', DATA_PROVENANCE)


In [ ]:
# ===== 3. Pure strategy constructors: explicit DV01 neutrality =====
def _normalize_notional_from_dv01_targets(g, q):
    """q = desired signed DV01 units. Converts to notionals and rescales gross notional to 1."""
    D=g['duration10'].to_numpy(float); q=np.asarray(q,float)
    w=q/D
    gross=np.sum(np.abs(w))
    if gross<=1e-12: return np.full(len(g),np.nan)
    w=w/gross
    # Hard identity: net DV01 must be zero up to numerical precision.
    assert abs(np.sum(w*D)) < 1e-10
    return w

def construct_rank_weights(g,k):
    g=g.dropna(subset=['tp_z_past','duration10']).sort_values('tp_z_past').copy()
    if len(g)<max(MIN_COUNTRIES,2*k): return None
    q=np.zeros(len(g)); q[:k]=-1.0/k; q[-k:]=1.0/k
    w=_normalize_notional_from_dv01_targets(g,q)
    g['weight']=w; g['dv01_weight']=w*g['duration10']
    return g

def construct_continuous_weights(g):
    g=g.dropna(subset=['tp_z_past','duration10']).copy()
    if len(g)<MIN_COUNTRIES: return None
    s=g['tp_z_past'].to_numpy(float); q=s-np.mean(s)
    if np.sum(np.abs(q))<1e-12: return None
    q=q/np.sum(np.abs(q))
    w=_normalize_notional_from_dv01_targets(g,q)
    g['weight']=w; g['dv01_weight']=w*g['duration10']
    return g

def form_portfolio_panel(panel, mode='rank', k=HEADLINE_K, signal='tp_z_past', target='rx'):
    rows=[]; weight_rows=[]
    for date,g0 in panel.groupby('date'):
        cols=['date','country','y10','r3m','slope','duration10',signal,target]
        g=g0[[c for c in cols if c in g0.columns]].dropna(subset=[signal,target,'duration10']).copy()
        # Map any requested signal into the constructor's canonical score column exactly once.
        # Do NOT rename the original signal afterward: that would create duplicate 'tp_z_past' columns.
        if signal!='tp_z_past':
            g['tp_z_past']=g[signal].astype(float)
        if mode=='rank':
            z=construct_rank_weights(g,k)
        elif mode=='continuous':
            z=construct_continuous_weights(g)
        else: raise ValueError(mode)
        if z is None: continue
        # recover the requested signal name if constructor renamed it
        ret=float(np.sum(z['weight']*z[target]))
        sig=float(np.sum(z['weight']*z['tp_z_past']))
        rows.append({'date':date,'ret12':ret,'signal_spread':sig,'n_countries':len(z),
                     'gross_notional':float(z['weight'].abs().sum()),
                     'net_notional':float(z['weight'].sum()),
                     'net_dv01':float(z['dv01_weight'].sum())})
        for _,r in z.iterrows():
            weight_rows.append({'date':date,'country':r['country'],'weight':r['weight'],
                                'dv01_weight':r['dv01_weight'],'duration10':r['duration10'],
                                'score':r['tp_z_past']})
    return pd.DataFrame(rows).set_index('date').sort_index(), pd.DataFrame(weight_rows)

def weight_turnover(weight_rows):
    W=weight_rows.pivot(index='date',columns='country',values='weight').fillna(0).sort_index()
    # one-way turnover: half of gross target-weight change
    to=0.5*W.diff().abs().sum(axis=1)
    if len(to): to.iloc[0]=0.5*W.iloc[0].abs().sum()
    return W,to

def summary_12m(port, weight_rows, cost_bp=0.0, fx_drag_bp=0.0):
    x=port['ret12'].dropna(); b,se,V,r2,n=nw_fit(x.to_numpy(),None,NW_LAG)
    W,to=weight_turnover(weight_rows)
    ann_turnover=float(12*to.reindex(x.index).mean())
    gross=float(x.mean())                     # each observation is already a 12m return
    net=gross-ann_turnover*(cost_bp/1e4)-fx_drag_bp/1e4
    tracks=[]
    for m in range(1,13):
        a=x[x.index.month==m]
        if len(a)>=8 and a.std(ddof=1)>1e-12: tracks.append(float(a.mean()/a.std(ddof=1)))
    return {'months':n,'gross_ann_mean':gross,'NW_t':float(b[0]/se[0]),
            'median_nonoverlap_SR':float(np.median(tracks)) if tracks else np.nan,
            'positive_tracks':f"{sum(t>0 for t in tracks)}/{len(tracks)}",
            'ann_turnover':ann_turnover,'net_ann_mean':net,
            'breakeven_bp':gross*1e4/ann_turnover if ann_turnover>0 else np.nan,
            'avg_abs_net_dv01':float(port['net_dv01'].abs().mean()),
            'avg_abs_net_notional':float(port['net_notional'].abs().mean())}


In [ ]:
# ===== 4. Build the two frozen V2 portfolios =====
headline, W_head = form_portfolio_panel(analysis,mode='rank',k=HEADLINE_K)
sens2, W_sens2 = form_portfolio_panel(analysis,mode='rank',k=SENSITIVITY_K)
continuous, W_cont = form_portfolio_panel(analysis,mode='continuous')

rows=[]
for name,p,w in [('rank_top3_bottom3_DV01',headline,W_head),('rank_top2_bottom2_DV01_sensitivity',sens2,W_sens2),('continuous_score_DV01',continuous,W_cont)]:
    s=summary_12m(p,w,0.0,0.0); rows.append({'strategy':name,**s})
perf=pd.DataFrame(rows).set_index('strategy')
print('Gross V2.3 results (12m overlapping formation returns; NW(12) inference)')
display(perf.round(5))

# Hard neutrality diagnostics
assert perf['avg_abs_net_dv01'].max() < 1e-10
print('DV01 neutrality identity: PASS')

In [ ]:
# ===== 5. Cost and FX-hedge stress grid =====
cost_rows=[]
for name,p,w in [('rank_top3_bottom3_DV01',headline,W_head),('continuous_score_DV01',continuous,W_cont)]:
    for c in COST_GRID_BP:
        for fx in FX_HEDGE_ANNUAL_DRAG_BP:
            s=summary_12m(p,w,c,fx)
            cost_rows.append({'strategy':name,'oneway_cost_bp':c,'fx_extra_annual_drag_bp':fx,
                              'gross_ann_mean':s['gross_ann_mean'],'net_ann_mean':s['net_ann_mean'],
                              'ann_turnover':s['ann_turnover'],'breakeven_bp':s['breakeven_bp']})
cost_table=pd.DataFrame(cost_rows)
display(cost_table.query("fx_extra_annual_drag_bp==0").round(5))

In [ ]:
# ===== 6. Pre-specified simple benchmarks: global duration, yield-level RV, slope/carry RV =====
# Past-only controls are the same objects used in notebook 09b.
for col in ['y10','slope']:
    if col+'_z_past' not in analysis:
        analysis[col+'_z_past']=analysis.groupby('country',group_keys=False)[col].transform(
            lambda s: pd.Series(past_only_z(s.to_numpy()),index=s.index))

# Global 10Y factor = equal-notional average of local-currency 10Y excess returns.
global10=analysis.groupby('date')['rx'].mean().rename('global10')

# Naive cross-country carry/value benchmarks use the exact same DV01-neutral rank constructor,
# changing only the signal. No benchmark tuning.
yield_ls, W_yield = form_portfolio_panel(analysis,mode='rank',k=HEADLINE_K,signal='y10_z_past')
slope_ls, W_slope = form_portfolio_panel(analysis,mode='rank',k=HEADLINE_K,signal='slope_z_past')

bench=pd.concat([global10,yield_ls['ret12'].rename('yield_level_LS'),slope_ls['ret12'].rename('slope_carry_LS')],axis=1)
display(bench.describe().T[['mean','std','count']])


In [ ]:
# ===== 7. Spanning / attribution regressions =====
def spanning(y, Xdf, label):
    d=pd.concat([y.rename('y'),Xdf],axis=1).dropna()
    b,se,V,r2,n=nw_fit(d['y'].to_numpy(),d[Xdf.columns].to_numpy(),NW_LAG)
    out={'model':label,'alpha_ann':float(b[0]),'alpha_t_NW':float(b[0]/se[0]),'R2':r2,'n':n}
    for j,c in enumerate(Xdf.columns,1): out['beta_'+c]=float(b[j])
    return out

span_rows=[]
for nm,p in [('rank_top3_bottom3_DV01',headline),('continuous_score_DV01',continuous)]:
    y=p['ret12']
    span_rows.append(spanning(y,bench[['global10']],nm+' ~ global10'))
    span_rows.append(spanning(y,bench[['global10','yield_level_LS','slope_carry_LS']],nm+' ~ global10 + yield/slope RV'))
spanning_table=pd.DataFrame(span_rows).set_index('model')
display(spanning_table.round(5))

print('Reading rule: positive alpha after global-duration and naive carry/value spanning is incremental MacroAnchor evidence;')
print('a null alpha means the tradable spread is explained by familiar sovereign-rate exposures even if the predictor slope remains significant.')

In [ ]:
# ===== 8. Stability by pre-specified regimes =====
def regime(d):
    d=pd.Timestamp(d)
    if d<=pd.Timestamp('2007-12-31'): return 'pre_2008'
    if d<=pd.Timestamp('2015-12-31'): return '2008_2015'
    return '2016_plus'

sub_rows=[]
for nm,p,w in [('rank_top3_bottom3_DV01',headline,W_head),('continuous_score_DV01',continuous,W_cont)]:
    for reg,g in p.assign(regime=[regime(x) for x in p.index]).groupby('regime'):
        ww=w[w['date'].isin(g.index)]
        s=summary_12m(g.drop(columns='regime'),ww,0.0,0.0)
        sub_rows.append({'strategy':nm,'regime':reg,**s})
subperiods=pd.DataFrame(sub_rows).set_index(['strategy','regime'])
display(subperiods[['months','gross_ann_mean','NW_t','median_nonoverlap_SR','ann_turnover']].round(5))

In [ ]:
# ===== 9. Diagnostics: country concentration, score dispersion, neutrality =====
def concentration(weight_rows):
    W=weight_rows.pivot(index='date',columns='country',values='weight').fillna(0).sort_index()
    hhi=(W.abs().div(W.abs().sum(axis=1),axis=0).pow(2).sum(axis=1))
    avg_abs=W.abs().mean().sort_values(ascending=False)
    return W,hhi,avg_abs

Wmat,hhi,country_exposure=concentration(W_head)
print('Headline median HHI:',float(hhi.median()),'| max:',float(hhi.max()))
display(country_exposure.rename('avg_abs_notional_weight').to_frame())

fig,ax=plt.subplots(1,2,figsize=(12,4))
headline['ret12'].dropna().cumsum().plot(ax=ax[0],title='Headline DV01-neutral cross-country spread: cumulative 12m-formation mean path')
ax[0].set_ylabel('cumulative spread (not a live wealth index)')
country_exposure.sort_values().plot.barh(ax=ax[1],title='Average absolute country notional weight')
plt.tight_layout(); plt.savefig(OUT_DIR/'figure_headline_diagnostics.png',bbox_inches='tight'); plt.show()

In [ ]:
# ===== 10. Export deterministic research bundle =====
perf.to_csv(OUT_DIR/'v2_performance.csv')
cost_table.to_csv(OUT_DIR/'v2_cost_fx_stress.csv',index=False)
spanning_table.to_csv(OUT_DIR/'v2_spanning.csv')
subperiods.to_csv(OUT_DIR/'v2_subperiods.csv')
W_head.to_csv(OUT_DIR/'v2_weights_headline.csv',index=False)
W_cont.to_csv(OUT_DIR/'v2_weights_continuous.csv',index=False)
country_exposure.to_csv(OUT_DIR/'v2_country_exposure.csv',header=['avg_abs_notional_weight'])

manifest={
    'label':'macroanchor_g10_10y_rv_v2',
    'panel_cutoff':str(PANEL_END.date()),
    'parent_signal':'MacroAnchor 10Y tp_z_past; unchanged',
    'headline':'top-3/bottom-3 explicit DV01-neutral',
    'sensitivity':'top-2/bottom-2 only; inherited from notebook 10',
    'second_spec':'continuous demeaned-score explicit DV01-neutral',
    'inference':'Newey-West lag 12 on overlapping 12m formation returns; 12 non-overlapping calendar-track Sharpe diagnostic',
    'fx_interpretation':'local excess return equals FX-hedged excess over base cash under CIP to first order; basis/forward execution not observed',
    'data_provenance':DATA_PROVENANCE,
    'no_search_statement':'No new predictor family, k-grid, threshold grid, or rebalance-frequency search introduced in V2.'
}
(OUT_DIR/'v2_manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
print('Exported to',OUT_DIR.resolve())

## V3 starts here — credit audit and geographic breadth

All V2.3 results above remain **the frozen original baseline**. No edits were made to V2.3 scoring, portfolio, cost, or inference code. Hereafter the analysis may restrict the sample to the CDS-coverage overlap. This does **not** create a new V2.3 result; the original 363-month headline remains distinct from any common-sample diagnostics.

For expanded countries the input must already provide MacroAnchor model objects (computed by the same parent method and independently audited); **no presumed data-vintage equivalence** is asserted for user-supplied additional rows.

In [ ]:
# ===== V3.2 Data contracts, as-of sovereign CDS, optional historical ratings and macro coverage =====
import unicodedata
from collections import Counter

V3_OUT=Path('macroanchor_g10_rv_v3_credit_outputs')
V3_OUT.mkdir(exist_ok=True)
V3_H=12
CREDIT_BAND_EDGES=[-np.inf,40.0,100.0,np.inf]
CREDIT_BAND_LABELS=['CDS_low_<40bp','CDS_mid_40_100bp','CDS_high_>=100bp']
MAX_MATCH_CDS_GAP_BP=25.0
MAX_CDS_STALENESS_DAYS=45
EXPECTED_PANEL_SHA=MODEL_PANEL_EXPECTED_SHA

COUNTRY_ALIASES={
    'us':'USA','usa':'USA','united states':'USA','united states of america':'USA','etats unis':'USA','etats-unis':'USA',
    'germany':'DEU','deutschland':'DEU','allemagne':'DEU','deu':'DEU',
    'united kingdom':'GBR','uk':'GBR','great britain':'GBR','royaume uni':'GBR','royaume-uni':'GBR','gbr':'GBR',
    'japan':'JPN','japon':'JPN','jpn':'JPN',
    'canada':'CAN','can':'CAN',
    'switzerland':'CHE','suisse':'CHE','che':'CHE',
    'sweden':'SWE','suede':'SWE','swe':'SWE',
    'norway':'NOR','norvege':'NOR','nor':'NOR',
    'australia':'AUS','australie':'AUS','aus':'AUS',
    'new zealand':'NZL','nouvelle zelande':'NZL','nzl':'NZL',
    'france':'FRA','fra':'FRA','italy':'ITA','italie':'ITA','ita':'ITA',
    'spain':'ESP','espagne':'ESP','esp':'ESP',
    'portugal':'PRT','prt':'PRT','belgium':'BEL','belgique':'BEL','bel':'BEL',
    'netherlands':'NLD','pays bas':'NLD','nld':'NLD','austria':'AUT','autriche':'AUT','aut':'AUT',
    'finland':'FIN','finlande':'FIN','fin':'FIN','ireland':'IRL','irlande':'IRL','irl':'IRL',
    'denmark':'DNK','danemark':'DNK','dnk':'DNK','south korea':'KOR','korea':'KOR','coree du sud':'KOR','kor':'KOR',
    'poland':'POL','pologne':'POL','pol':'POL','czech republic':'CZE','czechia':'CZE','tchequie':'CZE','cze':'CZE',
    'mexico':'MEX','mexique':'MEX','mex':'MEX','brazil':'BRA','bresil':'BRA','bra':'BRA',
}
def country_iso(v):
    s=''.join(c for c in unicodedata.normalize('NFKD',str(v)) if not unicodedata.combining(c))
    s=' '.join(s.lower().replace('_',' ').split())
    return COUNTRY_ALIASES.get(s,s.upper() if len(s)==3 else 'UNMATCHED:'+s)

def pick_column(df,options):
    x={str(c).strip().lower().replace(' ','_'):c for c in df.columns}
    for c in options:
        if c.lower() in x:return x[c.lower()]
    return None

def read_csv_or_parquet(path):
    if path.suffix.lower() in ('.parquet','.pq'):
        try:return pd.read_parquet(path)
        except ImportError as e:
            raise RuntimeError('Parquet support missing (pyarrow). Upload sovereign_cds_5y.csv instead.') from e
    return pd.read_csv(path)

def read_sovereign_cds(path):
    d=read_csv_or_parquet(path).copy()
    dt=pick_column(d,['date','observation_date','scraped_date','timestamp','datetime'])
    co=pick_column(d,['country','sovereign','entity','country_name','iso3','iso_code','country_code'])
    sp=pick_column(d,['cds5y_bp','cds_5y_bp','cds5y','cds_spread_bp','spread_bp','spread','last','mid','value','cds'])
    if any(v is None for v in (dt,co,sp)):
        raise ValueError('Historical CDS file must contain date, country and 5Y spread columns. Found '+str(list(d.columns)))
    mat=pick_column(d,['tenor','maturity','term'])
    if mat:
        d=d[d[mat].astype(str).str.upper().str.replace(' ','').isin(['5Y','5YR','5YEAR','5YEARS','60M'])]
    d=d.rename(columns={dt:'date',co:'country',sp:'cds5y_bp'})
    d['date']=pd.to_datetime(d['date'],errors='coerce',utc=True).dt.tz_localize(None)
    d['country']=d['country'].map(country_iso)
    d['cds5y_bp']=pd.to_numeric(d['cds5y_bp'],errors='coerce')
    d=d.dropna(subset=['date','cds5y_bp'])
    d=d[~d['country'].str.startswith('UNMATCHED:')]
    d=d[(d['cds5y_bp']>=0)&(d['cds5y_bp']<3000)]
    if d.empty:raise RuntimeError('No usable historical sovereign CDS spread observations after normalization.')
    # *Date* must be a historical observation date, not one scraped current snapshot.
    months=d['date'].dt.to_period('M').nunique()
    if months<12:raise RuntimeError(f'Only {months} CDS months; refuse to use current snapshots as historical data.')
    return d[['date','country','cds5y_bp']].sort_values(['country','date']).drop_duplicates(['country','date'],keep='last')

sovereign_cds=read_sovereign_cds(CDS_PATH)
print('Sovereign CDS imported:',len(sovereign_cds),'observations',sovereign_cds.country.nunique(),'countries',
      sovereign_cds.date.min().date(),'through',sovereign_cds.date.max().date())

# Optional additional country objects: explicitly supplied SAME PARENT-model outputs only.
expanded_files=[p for p in UPLOAD_PATHS if 'expanded_macroanchor_model_objects' in p.name.lower() and p.suffix.lower()=='.csv']
v3_model=model_panel.copy()
if expanded_files:
    if len(expanded_files)!=1:raise RuntimeError('Only one expanded model-objects file is allowed.')
    ext=pd.read_csv(expanded_files[0],parse_dates=['date'])
    needed={'date','country','tp_z_past','rx','y10','duration10','r3m','slope'}
    if not needed.issubset(ext.columns):raise ValueError('Expanded model panel must contain '+str(sorted(needed)))
    ext['country']=ext['country'].map(country_iso)
    ext=ext[~ext.country.str.startswith('UNMATCHED:')]
    if ext.duplicated(['date','country']).any():raise ValueError('Duplicate date-country keys in expanded model panel.')
    overlap=set(ext.country).intersection(model_panel.country)
    if overlap:
        print('Pre-existing countries in expanded panel excluded (original V2.3 always has priority):',sorted(overlap))
        ext=ext[~ext.country.isin(overlap)]
    v3_model=pd.concat([model_panel,ext],ignore_index=True).sort_values(['country','date'])
    print('Expanded countries added:',sorted(set(ext.country)))
else:
    print('No added sovereign MacroAnchor model objects uploaded: expansion is PLUMBED IN but not claimed executed.')

v3_model['country']=v3_model.country.map(country_iso)
v3_model['date']=pd.to_datetime(v3_model.date).dt.to_period('M').dt.to_timestamp('M')
v3_model=v3_model[v3_model.date<=PANEL_END].copy()
if v3_model.duplicated(['date','country']).any():raise ValueError('duplicate country-month keys in merged V3 panel')

def asof_country(base, hist, value_col, tolerance_days=45):
    # No backfill: only the most recent quote on/before the signal date.
    parts=[]
    for c, g in base.groupby('country',sort=True):
        h=hist[hist.country==c][['date',value_col]].sort_values('date')
        g=g.sort_values('date').copy()
        if not len(h):
            g[value_col]=np.nan;g[value_col+'_asof_date']=pd.NaT
        else:
            h=h.rename(columns={'date':value_col+'_asof_date'})
            g=pd.merge_asof(g,h,left_on='date',right_on=value_col+'_asof_date',
                            direction='backward',tolerance=pd.Timedelta(days=tolerance_days))
        parts.append(g)
    return pd.concat(parts,ignore_index=True).sort_values(['country','date'])

v3_panel=asof_country(v3_model,sovereign_cds,'cds5y_bp',MAX_CDS_STALENESS_DAYS)
valid=v3_panel.cds5y_bp.notna()
assert (v3_panel.loc[valid,'cds5y_bp_asof_date']<=v3_panel.loc[valid,'date']).all()

# Rating file is OPTIONAL and must represent historical dated ratings, not latest values broadcast backwards.
rating_files=[p for p in UPLOAD_PATHS if 'rating' in p.name.lower() and p.suffix.lower()=='.csv']
RATING_MAP={v:k for k,v in enumerate(['AAA','AA+','AA','AA-','A+','A','A-','BBB+','BBB','BBB-',
                                      'BB+','BB','BB-','B+','B','B-','CCC+','CCC','CCC-','CC','C','D'])}
MOODYS_EQUIV={'Aaa':'AAA','Aa1':'AA+','Aa2':'AA','Aa3':'AA-','A1':'A+','A2':'A','A3':'A-',
              'Baa1':'BBB+','Baa2':'BBB','Baa3':'BBB-','Ba1':'BB+','Ba2':'BB','Ba3':'BB-',
              'B1':'B+','B2':'B','B3':'B-','Caa1':'CCC+','Caa2':'CCC','Caa3':'CCC-','Ca':'CC'}

def rating_group(r):
    r=MOODYS_EQUIV.get(str(r),str(r)).strip().upper()
    if r=='AAA':return 'AAA'
    if r.startswith('AA'):return 'AA'
    if r.startswith('A') and not r.startswith('AA'):return 'A'
    if r.startswith('BBB'):return 'BBB'
    if r in RATING_MAP:return 'BB_or_lower'
    return None

v3_panel['rating_bucket']=None
if rating_files:
    if len(rating_files)!=1:raise RuntimeError('Only one dated historical ratings file accepted.')
    rat=pd.read_csv(rating_files[0]);dt=pick_column(rat,['date','rating_date','effective_date','asof_date']);co=pick_column(rat,['country','iso3','sovereign']);rc=pick_column(rat,['rating','global_rating','local_currency_rating'])
    if not all([dt,co,rc]):raise ValueError('Ratings history needs date,country,rating (not current-country table)')
    if 'scale' in rat.columns:
        scales=rat['scale'].astype(str).str.lower()
        if scales.str.contains('national').any():raise ValueError('National-scale ratings cannot be compared across countries.')
    rat=rat.rename(columns={dt:'date',co:'country',rc:'rating'})
    rat['date']=pd.to_datetime(rat.date,errors='coerce');rat['country']=rat.country.map(country_iso)
    rat['rating_bucket']=rat.rating.map(rating_group)
    rat=rat.dropna(subset=['date','rating_bucket'])[['date','country','rating_bucket']]
    rat=rat.sort_values(['country','date']).drop_duplicates(['country','date'],keep='last')
    if rat.date.dt.year.nunique()<2:raise ValueError('Ratings file is not a historical panel (only one year). No retrospective rating assignment allowed.')
    v3_panel=v3_panel.drop(columns=['rating_bucket'])
    v3_panel=asof_country(v3_panel,rat,'rating_bucket',450)
    print('Historical rating as-of coverage:',v3_panel.rating_bucket.notna().sum(),'country-months')
else:
    print('No historical sovereign rating file: rating-bucket test is explicitly SKIPPED.')

v3_panel['cds_bucket']=pd.cut(v3_panel.cds5y_bp,bins=CREDIT_BAND_EDGES,
                               labels=CREDIT_BAND_LABELS,right=False).astype(object)
v3_panel=v3_panel.dropna(subset=['tp_z_past','duration10']).copy()
v3_panel['rx']=pd.to_numeric(v3_panel['rx'],errors='coerce')
# Keep unknown future outcomes at formation: ranking must never depend on rx(t,t+12).
coverage=v3_panel.groupby('country').agg(rows=('date','size'),months_with_cds=('cds5y_bp','count'),
                                          first=('date','min'),last=('date','max'))
display(coverage)
if v3_panel.cds5y_bp.notna().sum()==0:
    raise RuntimeError('No overlapping sovereign CDS observations with frozen MacroAnchor panel. Check country mapping and dates.')


In [ ]:
# ===== V3.2 Pre-declared credit-stratified DV01-neutral portfolios =====
def candidate_pair(g, bucket_name, group_label, pair_gap_max=None):
    g=g.dropna(subset=['tp_z_past','duration10','cds5y_bp']).copy()
    if g.country.nunique()<2:return None
    # Pair maximizes ex-ante MacroAnchor-score difference only: no realized-return optimization.
    candidate=[]
    for l in range(len(g)):
        for s in range(len(g)):
            if l==s:continue
            a=g.iloc[l];b=g.iloc[s]
            if a.tp_z_past <= b.tp_z_past:continue
            gap=abs(a.cds5y_bp-b.cds5y_bp)
            if pair_gap_max is not None and gap>pair_gap_max:continue
            candidate.append((a.tp_z_past-b.tp_z_past,-gap,str(a.country),str(b.country)))
    if not candidate:return None
    _,_,long_c,short_c=sorted(candidate,reverse=True)[0]
    z=g[g.country.isin([long_c,short_c])].copy()
    z['q']=z.country.map({long_c:1.0,short_c:-1.0})
    w=_normalize_notional_from_dv01_targets(z,z.q.to_numpy(float))
    z['weight']=w
    z['dv01_weight']=w*z.duration10
    z['bucket']=str(bucket_name);z['type']=str(group_label)
    assert abs(z.dv01_weight.sum())<1e-10
    assert abs(z.weight.abs().sum()-1)<1e-10
    return z

def bucket_strategy(panel,bucket_col,tag):
    rows=[]
    all_w=[]
    for (d,b), g in panel.dropna(subset=[bucket_col]).groupby(['date',bucket_col],sort=True):
        z=candidate_pair(g,b,tag)
        if z is None:continue
        for _,r in z.iterrows():
            all_w.append({'date':d,'country':r.country,'bucket':str(b),
                          'weight_in_bucket':float(r.weight),'duration10':float(r.duration10),
                          'cds5y_bp':float(r.cds5y_bp),'rx':float(r.rx),'tp_z_past':float(r.tp_z_past),
                          'cds5y_bp_asof_date':r.cds5y_bp_asof_date})
    w=pd.DataFrame(all_w)
    if w.empty:return pd.DataFrame(),w,pd.DataFrame()
    # Predeclared equal gross-notional budget across active bucket portfolios each month.
    n_buckets=w.groupby('date').bucket.transform('nunique')
    w['weight']=w.weight_in_bucket/n_buckets
    w['dv01_weight']=w.weight*w.duration10
    assert w.groupby('date').dv01_weight.sum().abs().max()<1e-10
    assert np.allclose(w.groupby('date').weight.apply(lambda x:x.abs().sum()).values,1.0)
    w['weighted_ret']=w.weight*w.rx
    agg=w.groupby('date').agg(ret12=('weighted_ret','sum'),active_buckets=('bucket','nunique'),
                             countries=('country','nunique'),net_dv01=('dv01_weight','sum'),
                             gross_notional=('weight',lambda x:x.abs().sum()),
                             n_legs=('country','size'),n_realized_legs=('rx','count'))
    # Never let pandas' sum(skipna=True) masquerade as a realized full portfolio return.
    agg.loc[agg.n_realized_legs<agg.n_legs,'ret12']=np.nan
    buck=(w.assign(weighted_bucket_ret=w.weight_in_bucket*w.rx)
          .groupby(['date','bucket']).agg(ret12=('weighted_bucket_ret','sum'),
                                           nlegs=('country','size'),n_realized_legs=('rx','count'))).reset_index()
    buck.loc[buck.n_realized_legs<buck.nlegs,'ret12']=np.nan
    return agg.sort_index(),w.sort_values(['date','country']),buck

v3_cds, v3_cds_weights, v3_cds_by_bucket = bucket_strategy(v3_panel,'cds_bucket','CDS_band')
if len(v3_cds):
    print('CDS-bucket strategy formed months:',len(v3_cds),'| avg active buckets:',v3_cds.active_buckets.mean())
    display(v3_cds_by_bucket.groupby('bucket').agg(months=('ret12','size'),mean_ret12=('ret12','mean')))
else:
    print('No feasible within-CDS-band pairs. No CDS-bucket performance claimed.')

if v3_panel.rating_bucket.notna().any():
    v3_rating,v3_rating_weights,v3_rating_by_bucket=bucket_strategy(v3_panel,'rating_bucket','rating')
else:
    v3_rating,v3_rating_weights,v3_rating_by_bucket=pd.DataFrame(),pd.DataFrame(),pd.DataFrame()

# Separate named sensitivity: closest-credit pair under fixed 25bp max spread difference.
matched_rows=[]
for d,g in v3_panel.dropna(subset=['cds5y_bp']).groupby('date'):
    z=candidate_pair(g,'matched_25bp','matched_cds',MAX_MATCH_CDS_GAP_BP)
    if z is not None:
        for _,r in z.iterrows():
            matched_rows.append({'date':d,'country':r.country,'bucket':'matched_25bp',
                                 'weight':float(r.weight),'duration10':float(r.duration10),
                                 'cds5y_bp':float(r.cds5y_bp),'rx':float(r.rx),
                                 'cds5y_bp_asof_date':r.cds5y_bp_asof_date})
v3_matched_weights=pd.DataFrame(matched_rows)
if not v3_matched_weights.empty:
    t=v3_matched_weights.copy();t['weighted_ret']=t.weight*t.rx;t['dv01_weight']=t.weight*t.duration10
    v3_matched=t.groupby('date').agg(ret12=('weighted_ret','sum'),net_dv01=('dv01_weight','sum'),
                                      n_legs=('country','size'),n_realized_legs=('rx','count'))
    v3_matched.loc[v3_matched.n_realized_legs<v3_matched.n_legs,'ret12']=np.nan
else:v3_matched=pd.DataFrame()

# Absolute CDS-gap metrics are ex ante; a matching tolerance does NOT neutralize CS01.
def credit_exposure(w,name):
    if w.empty:return pd.DataFrame()
    out=[]
    for d,g in w.groupby('date'):
        obs={'date':d,'abs_DV01_weighted_CDS_level_bp':abs((g.weight*g.duration10*g.cds5y_bp).sum()),
             'net_notional_CDS_level_bp':(g.weight*g.cds5y_bp).sum(),
             'max_held_country_CDS_gap_bp':g.cds5y_bp.max()-g.cds5y_bp.min(),
             'nlegs':g.country.nunique()}
        out.append(obs)
    return pd.DataFrame(out).set_index('date')
exposure_cds=credit_exposure(v3_cds_weights,'cds')
exposure_rating=credit_exposure(v3_rating_weights,'rating')
exposure_matched=credit_exposure(v3_matched_weights,'matched')
if not exposure_cds.empty:display(exposure_cds.describe())


In [ ]:
# ===== V3.3 SAME-SAMPLE comparisons, costs, overlap-robust inference =====
def v3_mean_t(series):
    x=pd.Series(series).dropna().astype(float)
    if len(x)<30 or x.std(ddof=1)<1e-12:return {'n':len(x),'mean_12m':np.nan,'t_NW12':np.nan,'positive_tracks':np.nan,'median_nonoverlap_SR':np.nan}
    b,se,_,_,n=nw_fit(x.to_numpy(),None,NW_LAG)
    tracks=[]
    for m in range(1,13):
        s=x[x.index.month==m]
        if len(s)>=5 and s.std(ddof=1)>0:tracks.append((s.mean(),s.mean()/s.std(ddof=1)))
    return {'n':n,'mean_12m':float(x.mean()),'t_NW12':float(b[0]/se[0]) if se[0]>0 else np.nan,
            'positive_tracks':f'{sum(t[0]>0 for t in tracks)}/{len(tracks)}',
            'median_nonoverlap_SR':float(np.median([t[1] for t in tracks])) if tracks else np.nan}

# Restricted-country, frozen-rule baseline: same *eligible CDS-covered country set*, same dates.
# No outcome availability is used for ranking; returns are NA unless all held legs are observable.
restricted_records=[]
for date,g0 in v3_panel.dropna(subset=['cds5y_bp']).groupby('date'):
    g=g0.dropna(subset=['tp_z_past','duration10']).copy()
    z=construct_rank_weights(g,HEADLINE_K)
    if z is None:continue
    held=z[z.weight.abs()>1e-12]
    ret=float((held.weight*held.rx).sum()) if held.rx.notna().all() else np.nan
    restricted_records.append({'date':date,'ret12':ret,'country_universe_size':len(g),
                               'held_legs':len(held),'net_dv01':float((held.weight*held.duration10).sum())})
restricted_headline=(pd.DataFrame(restricted_records).set_index('date').sort_index()
                     if restricted_records else pd.DataFrame())

# Comparison sample: no claim that the 2008+ CDS-stratified run covers the entire 1960–2026 panel.
portfolios={'V2.3 original headline':headline.ret12,
            'V2.3 rule on CDS-eligible countries':restricted_headline.ret12 if not restricted_headline.empty else pd.Series(dtype=float),
            'V3 CDS buckets (equal-budget aggregate)':v3_cds.ret12 if not v3_cds.empty else pd.Series(dtype=float)}
if not v3_rating.empty:portfolios['V3 rating buckets']=v3_rating.ret12
if not v3_matched.empty:portfolios['V3 matched CDS pair <=25bp']=v3_matched.ret12
v3_comparisons=[]
for label, s in portfolios.items():
    info=v3_mean_t(s)
    v3_comparisons.append({'portfolio':label,'sample':'own_available',**info})

# Each V3 strategy is compared with ORIGINAL V2.3 on exactly the V3 strategy's dates.
for label,s in list(portfolios.items())[1:]:
    # Compare against V2.3 original and, separately, fixed V2.3 rule on CDS-eligible countries.
    base=restricted_headline.ret12 if not restricted_headline.empty and label!='V2.3 rule on CDS-eligible countries' else headline.ret12
    common=pd.concat([base.rename('v2'),s.rename('v3')],axis=1).dropna()
    if len(common)==0:continue
    v3_comparisons.append({'portfolio':'V2.3 eligible-country baseline on same dates as '+label if label!='V2.3 rule on CDS-eligible countries' else 'V2.3 original headline on same dates',
                           'sample':'common',**v3_mean_t(common.v2)})
    v3_comparisons.append({'portfolio':label,'sample':'common',**v3_mean_t(common.v3)})
v3_summary=pd.DataFrame(v3_comparisons)
display(v3_summary)

# Track actual aggregate weights when trading costs are assessed (not a proxy for realized execution).
def turnover_from_v3_weights(w):
    if w.empty:return pd.Series(dtype=float)
    W=w.groupby(['date','country']).weight.sum().unstack('country').fillna(0).sort_index()
    to=0.5*W.diff().abs().sum(axis=1)
    to.iloc[0]=0.5*W.iloc[0].abs().sum()
    return to
v3_turnover=[]
for label,w,port in [('CDS buckets',v3_cds_weights,v3_cds),('rating buckets',v3_rating_weights,v3_rating),('matched CDS pair',v3_matched_weights,v3_matched)]:
    if w.empty:continue
    to=turnover_from_v3_weights(w)
    valid_index=port.ret12.dropna().index.intersection(to.index)
    gross=port.loc[valid_index,'ret12'].mean()
    annual_turn=12*to.loc[valid_index].mean()
    for bp in [0,1,2,5,10,25]:
        v3_turnover.append({'portfolio':label,'cost_bp_per_one_way_unit':bp,'gross_mean_12m':gross,
                            'ann_turnover_proxy':annual_turn,
                            'net_mean_12m_after_proxy_cost':gross-annual_turn*bp/1e4})
v3_costs=pd.DataFrame(v3_turnover)
if not v3_costs.empty:display(v3_costs)


In [ ]:
# ===== V3.4 Observed sovereign-CDS 12m changes and optional corporate credit-cycle controls =====
# Important: -duration * CDS change is a diagnostic spread-DV01 approximation, NOT a CDS hedge or cash-bond OAS P&L.
cds_panel_for_changes=v3_panel[['date','country','cds5y_bp']].drop_duplicates(['date','country']).copy()
forward_cds=cds_panel_for_changes.rename(columns={'cds5y_bp':'cds_5y_bp_at_t_plus_12'}).copy()
forward_cds['date']=forward_cds.date-pd.offsets.MonthEnd(V3_H)
cds_panel_for_changes=cds_panel_for_changes.merge(forward_cds,on=['date','country'],how='left',validate='one_to_one')
cds_panel_for_changes['delta_cds_12m_bp']=(cds_panel_for_changes.cds_5y_bp_at_t_plus_12-
                                          cds_panel_for_changes.cds5y_bp)

def attribution_panel(w):
    if w.empty:return pd.DataFrame()
    tmp=w.merge(cds_panel_for_changes[['date','country','delta_cds_12m_bp']],on=['date','country'],how='left',validate='many_to_one')
    tmp['implied_credit_component']=(-tmp.weight*tmp.duration10*tmp.delta_cds_12m_bp/1e4)
    out=tmp.groupby('date').apply(lambda g:pd.Series({
        'all_legs_credit_changes_available':bool(g.delta_cds_12m_bp.notna().all()),
        'rate_credit_proxy_pnl_12m':g.implied_credit_component.sum() if g.delta_cds_12m_bp.notna().all() else np.nan,
        'credit_level_proxy_bp':(g.weight*g.duration10*g.cds5y_bp).sum(),
        'nlegs':g.country.nunique()}),include_groups=False)
    return out
v3_attr_cds=attribution_panel(v3_cds_weights)
v3_attr_rating=attribution_panel(v3_rating_weights)
if not v3_attr_cds.empty:
    d=v3_cds.join(v3_attr_cds,how='left')
    d['rates_residual_illustrative']=d.ret12-d.rate_credit_proxy_pnl_12m
    print('CDS bucket attribution: ONLY proxy, usable complete-change months=',d.rate_credit_proxy_pnl_12m.notna().sum())
    print(d[['ret12','rate_credit_proxy_pnl_12m','rates_residual_illustrative','credit_level_proxy_bp']].mean())

# Optional corporate OAS / Moody's Aaa-Baa: aggregate stress control only, not sovereign CDS proxy.
corporate_files=[p for p in UPLOAD_PATHS if ('corporate_credit' in p.name.lower() or 'oas' in p.name.lower()) and p.suffix.lower()=='.csv']
corpo_diag=pd.DataFrame()
if corporate_files:
    if len(corporate_files)!=1:raise RuntimeError('Only one corporate credit factors CSV accepted.')
    corp=pd.read_csv(corporate_files[0]);date_col=pick_column(corp,['date','observation_date'])
    if date_col is None:raise ValueError('corporate_credit_factors CSV missing date')
    corp=corp.rename(columns={date_col:'date'});corp['date']=pd.to_datetime(corp.date,errors='coerce')
    corp=corp.dropna(subset=['date']).set_index('date').sort_index().resample('ME').last()
    # Moody's BAA and AAA levels are percentage yields, not OAS. Compute BAA-AAA in basis points.
    if {'BAA','AAA'}.issubset(corp.columns):
        corp['moodys_BAA_minus_AAA_bp']=(pd.to_numeric(corp['BAA'],errors='coerce')-
                                         pd.to_numeric(corp['AAA'],errors='coerce'))*100
    others=[c for c in corp.columns if ('OAS' in str(c).upper() or str(c).upper().startswith(('BAMLC','BAMLH')) or c=='moodys_BAA_minus_AAA_bp')]
    for c in others:
        vals=pd.to_numeric(corp[c],errors='coerce')
        future=vals.shift(-12)
        corpo_diag[c+'_12m_change']=future-vals
    if len(v3_cds):
        corpo_diag=corpo_diag.reindex(v3_cds.index)
        print('Corporate credit-cycle factor coverage (12m change):')
        display(corpo_diag.notna().sum())
else:
    print('Corporate OAS/Moody\'s factors not supplied: optional global-credit-cycle attribution skipped.')


In [ ]:
# ===== V3.4b Optional corporate OAS / Moody's credit cycle spanning =====
# Attribution only: contemporaneous 12m credit spread change is an observed future factor,
# NOT a predictor available at portfolio-formation time.
cycle_spans=[]
credit_cycle_candidates=['moodys_BAA_minus_AAA_bp_12m_change',
                         'BAMLC0A4CBBB_12m_change','BAMLC0A0CM_12m_change']
chosen_cycle=next((c for c in credit_cycle_candidates
                   if c in corpo_diag.columns and corpo_diag[c].notna().sum()>=48),None)
if chosen_cycle is None:
    print('No corporate credit-cycle factor with >=48 12m observations: spanning skipped.')
else:
    cycle=corpo_diag[chosen_cycle].rename('credit_cycle_change')
    candidates=[('V3_CDS_bucket_aggregate',v3_cds.ret12 if not v3_cds.empty else pd.Series(dtype=float)),
                ('V2_original',headline.ret12),
                ('V2_restricted',restricted_headline.ret12 if not restricted_headline.empty else pd.Series(dtype=float))]
    for label,y in candidates:
        joined=pd.concat([y.rename('ret12'),global10.rename('global10'),cycle],axis=1).dropna()
        if len(joined)<48:continue
        b,se,V,r2,n=nw_fit(joined.ret12.to_numpy(),joined[['global10','credit_cycle_change']].to_numpy(),NW_LAG)
        cycle_spans.append({'strategy':label,'credit_factor':chosen_cycle,'n':n,
            'alpha_12m':float(b[0]),'alpha_t_NW12':float(b[0]/se[0]) if se[0]>0 else np.nan,
            'beta_global10':float(b[1]),'beta_corporate_cycle':float(b[2]),'R2':float(r2)})
    if cycle_spans:display(pd.DataFrame(cycle_spans))
    else:print('Corporate spread series present, but no >=48-month aligned strategy sample.')
v3_credit_cycle_spanning=pd.DataFrame(cycle_spans)


In [ ]:
# ===== V3.5 Export transparent reproducibility bundle =====
import hashlib
import shutil
if Path('investing_raw_cds_sources.json').exists():
    shutil.copy2('investing_raw_cds_sources.json',V3_OUT/'investing_raw_cds_sources.json')
input_manifest=[]
for path in UPLOAD_PATHS:
    h=hashlib.sha256(path.read_bytes()).hexdigest() if path.exists() else None
    input_manifest.append({'name':path.name,'sha256':h,'bytes':path.stat().st_size if path.exists() else None})

def emit_df(df,name,index=False):
    if isinstance(df,pd.DataFrame) and not df.empty:df.to_csv(V3_OUT/name,index=index)

emit_df(coverage,'country_coverage.csv',True)
emit_df(v3_summary,'v3_common_sample_comparisons.csv')
emit_df(restricted_headline,'v2_3_rule_restricted_country_universe.csv',True)
emit_df(v3_cds_weights,'v3_cds_buckets_weights.csv')
emit_df(v3_cds_by_bucket,'v3_cds_bucket_returns.csv')
emit_df(v3_rating_weights,'v3_ratings_weights.csv')
emit_df(v3_rating_by_bucket,'v3_rating_bucket_returns.csv')
emit_df(v3_matched_weights,'v3_cds_matched_pair_weights.csv')
emit_df(v3_costs,'v3_cost_grid.csv')
emit_df(v3_attr_cds,'v3_cds_spread_change_attribution_proxy.csv',True)
emit_df(v3_attr_rating,'v3_rating_spread_change_attribution_proxy.csv',True)
emit_df(corpo_diag,'v3_corporate_oas_global_controls.csv',True)
emit_df(v3_credit_cycle_spanning,'v3_corporate_credit_cycle_spanning.csv')
emit_df(v3_panel[['date','country','cds5y_bp','cds5y_bp_asof_date','cds_bucket','rating_bucket']],
        'v3_asof_credit_panel_audit.csv')
manifest={'status':'V3 COMPUTED on user-supplied data; not independent test execution by package author',
          'original_v2_3_panel_SHA_required':EXPECTED_PANEL_SHA,
          'original_panel_SHA_observed':DATA_PROVENANCE.get('sha256'),
          'data_provenance_v2':DATA_PROVENANCE,'input_uploads':input_manifest,
          'cds_proxy_only':'5Y sovereign CDS used for stratification. No true 10Y cash bond CS01 hedge.',
          'rating_backfill':'No current ratings backfilled; only point-in-time historical dates allowed',
          'fx_implementation':'No historical FX forward hedging; returns remain local yield/funding proxies',
          'cds_bins_bp':CREDIT_BAND_EDGES[1:-1], 'cds_pair_gap_bp':MAX_MATCH_CDS_GAP_BP,
          'macroanchor_frozen':'V2.3 original predictor and rank weights were not retuned',
          'N_cds_available_after_signal_filter':int(v3_panel.cds5y_bp.notna().sum()),
          'N_countries_with_overlap':int(v3_panel.loc[v3_panel.cds5y_bp.notna(),'country'].nunique()),
          'N_expanded_countries':int(v3_model.country.nunique()-model_panel.country.nunique())}
(V3_OUT/'manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
print('V3 outputs:',V3_OUT.resolve())
print('FILES:',[p.name for p in V3_OUT.iterdir()])


## Reading rule (fixed before V3 execution)

**Do not promote the best-looking bucket, country set, or matching threshold to the new headline.** Compare every V3 construction with frozen V2.3 on *exactly the same months and available countries*. A positive V3 gross return alone is not evidence of incremental alpha; evaluate coverage, concentration, cost, loss to missing-data selection, and factor controls. CDS-bucketing is an imperfect risk-matching procedure, **not CS01-neutralization**. A first difference in sovereign CDS can be used for ex-post attribution, but neither a 5Y CDS quote nor US corporate OAS is a valid exact decomposition of local-currency 10Y government yields. For expanded-country inference, check newly introduced monetary and sovereign credit regimes explicitly.